In [1]:
import duckdb

con = duckdb.connect()

amazon = con.execute("""
    SELECT *
    FROM read_parquet(
        '../data/processed/amazon_conversations.parquet'
    )
""").df()

amazon.shape

(373312, 7)

In [2]:
amazon.head(20)

,conversation_id,depth,tweet_id,author_id,role,created_at,text
0,1000119,0,1000119,356779,customer,2017-10-22 22:51:06+05:30,Irritated on principle. I spent 2+ hrs looking...
1,1000119,1,1000117,AmazonHelp,support,2017-10-22 22:57:10+05:30,@356779 I'm sorry for the poor experience! Whe...
2,1000119,2,1000118,356779,customer,2017-10-22 23:15:55+05:30,@AmazonHelp It's all good now 😊 I chatted in &...
3,1000119,3,1000120,AmazonHelp,support,2017-10-22 23:22:00+05:30,@356779 Thanks for letting us know! We're alwa...
4,1000119,4,1000121,356779,customer,2017-10-22 23:26:09+05:30,@AmazonHelp https://t.co/q3YI6TgBjO
5,1000123,0,1000123,356780,customer,2017-10-22 22:51:46+05:30,@115830 I just inadvertently bought a Kindle b...
6,1000123,1,1000122,AmazonHelp,support,2017-10-22 22:57:00+05:30,@356780 Oh no! Please give us a call here: htt...
7,1000125,0,1000125,356781,customer,2017-10-22 22:46:40+05:30,@119625 Please upload south movies in hindi audio
8,1000125,1,1000124,AmazonHelp,support,2017-10-22 22:56:41+05:30,"@356781 I understand your concern, Nehal. I'll..."
9,1000127,0,1000127,356782,customer,2017-10-22 22:48:55+05:30,@43895 watching Loveless on Amazon prime cuz o...


In [3]:
amazon["role"].value_counts()

role
customer    202994
support     170318
Name: count, dtype: int64

In [4]:
customer_messages = amazon[
    amazon["role"] == "customer"
].copy()

print("Customer messages:", len(customer_messages))

Customer messages: 202994


In [5]:
customer_messages[
    ["conversation_id", "created_at", "text"]
].sample(
    50,
    random_state=42
)

,conversation_id,created_at,text
187538,2526960,2017-11-16 09:56:22+05:30,@115821 I ordered a popsocket for my sons 13th...
277074,395980,2017-10-10 00:32:20+05:30,@AmazonHelp Ok gracias otra vez!
283104,42576,2017-11-01 21:14:27+05:30,@115850 when you say cash on delivery is it ho...
77390,151988,2017-11-24 21:30:58+05:30,@AmazonHelp https://t.co/N8MVG9Zfyb
289581,453361,2017-12-01 04:02:47+05:30,@AmazonHelp Heyyy @AmazonHelp 😰
260903,321194,2017-10-07 19:05:09+05:30,@AmazonHelp Can’t. On my way to catch a flight...
97258,1672266,2017-11-06 13:50:13+05:30,Amazonで注文してたウルトラカプセルが届いたぜ。 https://t.co/iVaJME...
28099,1207990,2017-10-25 15:34:18+05:30,"@AmazonHelp Thanks for quick response, I will"
256294,302253,2017-10-07 03:29:38+05:30,"@AmazonHelp 2/2 That is extremely problematic,..."
17232,1124530,2017-10-24 14:50:51+05:30,@AmazonHelp Amazon mailed me on 7th august tha...


In [6]:
random_state=42

In [7]:
import duckdb

con = duckdb.connect()

support_accounts_in_amazon = con.execute("""
    SELECT
        author_id AS support_account,
        COUNT(*) AS message_count
    FROM read_parquet(
        '../data/processed/amazon_conversations.parquet'
    )
    WHERE role = 'support'
    GROUP BY author_id
    ORDER BY message_count DESC
""").df()

support_accounts_in_amazon

,support_account,message_count
0,AmazonHelp,169714
1,UPSHelp,322
2,Tesco,172
3,XboxSupport,14
4,hulu_support,11
5,SpotifyCares,10
6,JetBlue,9
7,ChaseSupport,8
8,BofA_Help,8
9,AskeBay,7


In [8]:
contaminated_conversations = con.execute("""
    SELECT
        conversation_id,
        COUNT(*) AS non_amazon_support_messages
    FROM read_parquet(
        '../data/processed/amazon_conversations.parquet'
    )
    WHERE role = 'support'
      AND author_id != 'AmazonHelp'
    GROUP BY conversation_id
    ORDER BY non_amazon_support_messages DESC
""").df()

print("Contaminated conversations:",
      len(contaminated_conversations))

contaminated_conversations.head(20)

Contaminated conversations: 310


,conversation_id,non_amazon_support_messages
0,397889,167
1,438763,16
2,463360,12
3,888183,9
4,191780,9
5,1363832,5
6,1127350,5
7,1741720,5
8,2197672,4
9,1863309,4


In [9]:
clean_amazon = con.execute("""
    WITH contaminated AS (
        SELECT DISTINCT conversation_id
        FROM read_parquet(
            '../data/processed/amazon_conversations.parquet'
        )
        WHERE role = 'support'
          AND author_id != 'AmazonHelp'
    )

    SELECT *
    FROM read_parquet(
        '../data/processed/amazon_conversations.parquet'
    )
    WHERE conversation_id NOT IN (
        SELECT conversation_id
        FROM contaminated
    )
""").df()

print("Clean shape:", clean_amazon.shape)

print(
    clean_amazon["role"].value_counts()
)

print(
    "Conversations:",
    clean_amazon["conversation_id"].nunique()
)

Clean shape: (370687, 7)
role
customer    201572
support     169115
Name: count, dtype: int64
Conversations: 82246


In [10]:
import duckdb

con = duckdb.connect()

conversation_397889 = con.execute("""
    SELECT
        conversation_id,
        depth,
        tweet_id,
        author_id,
        role,
        created_at,
        text
    FROM read_parquet(
        '../data/processed/amazon_conversations.parquet'
    )
    WHERE conversation_id = '397889'
    ORDER BY created_at
""").df()

conversation_397889.shape

(425, 7)

In [11]:
conversation_397889["author_id"].value_counts().head(20)

author_id
Tesco     167
525697     17
531486      9
525708      7
524415      6
374104      4
512760      4
522483      4
626265      4
479023      3
366324      3
150460      3
524199      3
482307      3
639922      3
210210      3
525700      3
525710      3
481057      2
482788      2
Name: count, dtype: int64

In [12]:
conversation_397889[
    ["created_at", "author_id", "role", "text"]
].head(20)

,created_at,author_id,role,text
0,2017-11-02 23:11:07+05:30,Tesco,support,The definitive WWII campaign at an unbeatable ...
1,2017-11-04 12:57:57+05:30,479023,customer,@Tesco £43.51 ;)
2,2017-11-04 12:58:43+05:30,Tesco,support,"@479023 Good morning RJ, is there anything I c..."
3,2017-11-04 12:59:05+05:30,479023,customer,@Tesco No thank you.
4,2017-11-04 13:01:25+05:30,Tesco,support,"@479023 Okay no probs, have a good weekend. :)..."
5,2017-11-04 13:07:37+05:30,479024,customer,@Tesco @479023 Sexy you Lara
6,2017-11-04 15:00:20+05:30,593315,customer,@Tesco Paying the money to just look at a load...
7,2017-11-04 15:17:52+05:30,481057,customer,@Tesco I pre-ordered it and payed £46
8,2017-11-04 15:40:23+05:30,Tesco,support,"@593315 Hi there, can you give us some details..."
9,2017-11-04 15:44:34+05:30,Tesco,support,@481057 Hi James. Can you elaborate on what yo...


In [13]:
conversation_397889[
    ["created_at", "author_id", "role", "text"]
].tail(20)

,created_at,author_id,role,text
405,2017-11-10 14:31:05+05:30,210210,customer,@Tesco So if your suppliers told you to market...
406,2017-11-10 15:48:21+05:30,Tesco,support,@210210 The retailer have given this descripti...
407,2017-11-10 19:03:29+05:30,525697,customer,"@525700 @525699 @Tesco shut the fuck up idiot,..."
408,2017-11-10 20:59:24+05:30,656408,customer,@Tesco How come its £44 but i got charged £48?
409,2017-11-10 23:02:08+05:30,525700,customer,@525697 @525699 @Tesco Nibble nibble 😩😩
410,2017-11-10 23:46:54+05:30,525697,customer,@525700 @525699 @Tesco not even ganna mention ...
411,2017-11-11 00:59:17+05:30,525710,customer,@626265 @Tesco Mate let’s cancel England vs Ge...
412,2017-11-11 01:59:58+05:30,Tesco,support,"@656408 Hi Kieron, can you please DM your full..."
413,2017-11-11 02:59:36+05:30,659560,customer,@Tesco Bought it this evening in store but pay...
414,2017-11-11 04:00:21+05:30,525700,customer,@525697 @525699 @Tesco Getting personal😩😩😩 nib...


In [14]:
conversation_397889.groupby(
    ["author_id", "role"]
).size().sort_values(
    ascending=False
)

author_id  role    
Tesco      support     167
525697     customer     17
531486     customer      9
525708     customer      7
524415     customer      6
                      ... 
496703     customer      1
496477     customer      1
495804     customer      1
494961     customer      1
517128     customer      1
Length: 157, dtype: int64

In [15]:
for conversation_id in ["397889", "438763", "463360"]:
    print("=" * 80)
    print("CONVERSATION:", conversation_id)
    print("=" * 80)

    summary = con.execute(f"""
        SELECT
            author_id,
            role,
            COUNT(*) AS messages
        FROM read_parquet(
            '../data/processed/amazon_conversations.parquet'
        )
        WHERE conversation_id = '{conversation_id}'
        GROUP BY author_id, role
        ORDER BY messages DESC
    """).df()

    print(summary)

CONVERSATION: 397889
    author_id      role  messages
0       Tesco   support       167
1      525697  customer        17
2      531486  customer         9
3      525708  customer         7
4      524415  customer         6
..        ...       ...       ...
152    494961  customer         1
153    486873  customer         1
154    244187  customer         1
155    517128  customer         1
156    516564  customer         1

[157 rows x 3 columns]
CONVERSATION: 438763
     author_id      role  messages
0      UPSHelp   support        16
1       323800  customer         7
2       767260  customer         2
3   AmazonHelp   support         2
4       461516  customer         2
5       591947  customer         2
6       748556  customer         1
7       519958  customer         1
8       303339  customer         1
9       744241  customer         1
10      115817  customer         1
11      151261  customer         1
12      137732  customer         1
13      783124  customer         1
1